# 02. Spatial cross-validation of top settings

## Setup

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.training import tuning
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Inputs

In [ ]:
grid_run = ""  # Optional pipeline_outputs/tuning/grid_<id>; blank uses the latest grid run
top_count = 3
grid_run = Path(grid_run) if grid_run else tuning.latest_run(root, "grid")
grid_inputs = pd.read_json(grid_run / "inputs.json", typ="series")
features_path, development = tuning.load_development(root, grid_inputs["development_features"])
grid_records = tuning.load_records(grid_run, "validation_grid")
print("Grid run:", grid_run)
print("Development features:", features_path)

## Candidate settings

In [ ]:
n_estimators = int(grid_inputs["n_estimators"])
candidates = {}
for cohort in tuning.COHORTS:
    baseline = [{**tuning.BASELINE, "n_estimators": n_estimators, "min_samples_leaf": leaf} for leaf in (2, 4)]
    settings = baseline + tuning.top_settings(grid_records, cohort, top_count)
    unique = []
    for params in settings:
        if params not in unique:
            unique.append(params)
    candidates[cohort] = unique
pd.DataFrame([{"cohort": c, **p} for c, values in candidates.items() for p in values])

## Run spatial folds

In [ ]:
run = tuning.new_run(root, "spatial", features_path, grid_run=str(grid_run), top_count=top_count)
records = []
for cohort, settings in candidates.items():
    for params in settings:
        pooled, folds = tuning.spatial_cross_validation(development, cohort, params)
        baseline = params["max_features"] == "sqrt" and params["max_samples"] == .5 and params["max_depth"] is None \
            and params["min_samples_leaf"] in (2, 4)
        validation = next(r["macro_f1"] for r in grid_records if r["cohort"] == cohort
                          and all(r[k] == params[k] for k in tuning.PARAMETERS))
        records.append({"cohort": cohort, **params, "baseline_setting": baseline,
                        "validation_macro_f1": validation,
                        "spatial_macro_f1": pooled["macro_f1"],
                        "spatial_balanced_accuracy": pooled["balanced_accuracy"],
                        "fold_macro_f1_min": min(f for f in folds if f is not None) if any(folds) else None,
                        "fold_macro_f1_max": max(f for f in folds if f is not None) if any(folds) else None,
                        **{f"spatial_{crop}_f1": v["f1"] for crop, v in pooled["per_crop"].items()}})
        print(cohort, params["max_features"], params["min_samples_leaf"], params["max_samples"], params["max_depth"],
              f"spatial macro-F1 {pooled['macro_f1']:.3f}", flush=True)
print("Saved:", tuning.save_records(run, "spatial_cv", records))

## Results table

In [ ]:
spatial = pd.DataFrame(records)
spatial["max_depth"] = spatial["max_depth"].astype(str)
spatial["max_samples"] = spatial["max_samples"].astype(str)
display(spatial.sort_values(["cohort", "spatial_macro_f1"], ascending=[True, False]).round(3))

## Figure

In [ ]:
figure, axes = plt.subplots(1, len(tuning.COHORTS), figsize=(5 * len(tuning.COHORTS), 4), sharey=True)
for axis, cohort in zip(axes, tuning.COHORTS):
    table = spatial[spatial.cohort == cohort].reset_index(drop=True)
    labels = [f"{r.max_features}/{r.min_samples_leaf}/{r.max_samples}/{r.max_depth}" for r in table.itertuples()]
    positions = range(len(table))
    axis.bar([p - .2 for p in positions], table.validation_macro_f1, width=.4, label="validation")
    axis.bar([p + .2 for p in positions], table.spatial_macro_f1, width=.4, label="spatial CV")
    axis.set_xticks(list(positions), labels, rotation=60, ha="right", fontsize=8)
    axis.set_title(cohort)
axes[0].set_ylabel("Parcel macro-F1")
axes[-1].legend()
plt.tight_layout()
figure.savefig(run / "spatial_cv.png", dpi=150)
display(figure)
plt.close(figure)
print("Run folder:", run)